<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用JAX 和Flax 微調循環Gemma

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/recurrentgemma/recurrentgemma_jax_finetune"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/recurrentgemma/recurrentgemma_jax_finetune.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/recurrentgemma/recurrentgemma_jax_finetune.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemma%2Fcookbook%2Fmain%2Fdocs%2Frecurrentgemma%2Frecurrentgemma_jax_finetune.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/recurrentgemma/recurrentgemma_jax_finetune.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

本教學示範如何使用 [Google DeepMind 的 `recurrentgemma` library](https://github.com/google-deepmind/recurrentgemma)、[JAX](https://jax.readthedocs.io](https://github.com/google-deepmind/recurrentgemma)、[JAX](https://jax.readthedocs.io](@@P002@@)、[JAX](https://jax.readthedocs.io)(高效能數值計算)微調英文-法文翻譯任務的 [Recurrent@@P0011 In 模型library)、[Flax](https://flax.readthedocs.io)（基於JAX的神經網路library）、[Ch ex](https://chex.readthedocs.io/en/latest/)（用於編寫可靠JAX程式碼的library實用程式）、[Optax](https://optax.readthedocs.io/en/latest/) （基於JAX的梯度處理和最佳化library），以及[MTNT（嘈雜文字的機器翻譯）dataset]（https://arxiv.org/abs/1809.00388）。雖然在這個notebook中沒有直接使用Flax，但是Flax被用來創建Gemma。
`recurrentgemma` library 是用 JAX、Flax、[Orbax](https://orbax.readthedocs.io/)（基於 JAX 的 library 用於訓練實用程序，如 @@P007@@ 的 library 用於訓練實用程序，如 @@P007@@ 的 library 用於訓練程序，如 @@Pce.編寫的（tokenizer/detokenizer library）。
此 notebook 可以在具有 T4 GPU 的 Google Colab 上執行（轉至 **編輯** > **notebook 設定** > 在 **硬體加速器** 下選擇 **T4 GPU**）。

## 設定

以下部分介紹了準備 notebook 以使用 RecurrentGemma 模型的步驟，包括模型存取、獲取 API 金鑰以及設定 notebook runtime。

### 為Gemma 設定Kaggle 存取權限

要完成本教學，您首先需要遵循類似[Gemma 設定](https://ai.google.dev/gemma/docs/setup) 的設定說明，但有些例外：
* 在 [kaggle.com](https://www.kaggle.com/models/google/recurrentgemma) 上造訪 RecurrentGemma（而非 Gemma）。
* 選擇具有足夠資源的 Colab runtime 來執行 RecurrentGemma 模型。
* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

完成 RecurrentGemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。
### 設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。當 prompted 並選擇「授予存取權限？」時訊息，同意提供secret存取。

In [ ]:
import os
from google.colab import userdata # `userdata` is a Colab API.

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 安裝 `recurrentgemma` library

免費Colab硬體加速目前*不足以*執行此notebook。如果您使用的是 [Colab Pay As You Go 或 Colab Pro](https://colab.research.google.com/signup)，請點選 **編輯** > **notebook 設定** > 選擇 **A100 GPU** > **儲存** 以啟用硬體加速。
接下來，您需要從 [`github.com/google-deepmind/recurrentgemma`](https://github.com/google-deepmind/recurrentgemma) 安裝 Google DeepMind `recurrentgemma` library。如果您收到有關「pip 的依賴解析器」的錯誤，通常可以忽略它。
**附註：**透過安裝`recurrentgemma`，您也將安裝[`flax`](https://flax.readthedocs.io)、核心[`jax`](https://jax.readthedocs.io](@@P006@@)、核心[`jax`](https://jax.readthedocs.io](`optax`](@@P0008@0)（基於處理的梯度@P0103@P@P00008@P000008 [`orbax`](https://orbax.readthedocs.io/) 和 [`sentencepiece`](https://github.com/google/sentencepiece)。

In [ ]:
!pip install -q git+https://github.com/google-deepmind/recurrentgemma.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.4 MB/s eta 0:00:00


### 導入庫

這個notebook使用[Flax](https://flax.readthedocs.io)（用於神經網路），核心[JAX](https://jax.readthedocs.io)，[SentencePiece](https://github.com/google/sentencepiece)(用於@@P0007)（用於@ library 用於編寫可靠的JAX 程式碼的實用程式）、[Optax](https://optax.readthedocs.io/en/latest/)（梯度處理和最佳化library）和 TensorFlow dataset。

In [ ]:
import pathlib
from typing import Any, Mapping, Iterator
import enum
import functools

import chex
import jax
import jax.numpy as jnp
import optax

import tensorflow as tf
import tensorflow_datasets as tfds

import sentencepiece as spm

from recurrentgemma import jax as recurrentgemma

## 載入循環Gemma模型

1. 使用 [`kagglehub.model_download`](https://github.com/Kaggle/kagglehub/blob/bddefc718182282882b72f814d407d89e5d178c4/src/kagglehub/models.py#L12) 載入 RecurrentGemma 模型，此模型採用三個參數：

- `handle`：來自Kaggle的模型句柄
- `path`：（可選字串）本地路徑
- `force_download`：（可選布林值）強制重新下載模型

**注意：** 請注意，RecurrentGemma 2B (IT) 型號的大小約為 3.85Gb。

In [ ]:
RECURRENTGEMMA_VARIANT = '2b-it' # @param ['2b', '2b-it'] {type:"string"}

In [ ]:
import kagglehub

RECURRENTGEMMA_PATH = kagglehub.model_download(f'google/recurrentgemma/flax/{RECURRENTGEMMA_VARIANT}')

100%|██████████| 3.85G/3.85G [00:50<00:00, 81.5MB/s]
Extracting model files...


In [ ]:
print('RECURRENTGEMMA_VARIANT:', RECURRENTGEMMA_VARIANT)

RECURRENTGEMMA_VARIANT: 2b-it


**注意：** 上面輸出的路徑是模型權重和tokenizer本地保存的位置，稍後您將需要它們。

2. 檢查模型權重和tokenizer的位置，然後設定路徑變數。 tokenizer 目錄將位於您下載模型的主目錄中，而模型權重將位於子目錄中。例如：

- `tokenizer.model` 檔案將位於 `/LOCAL/PATH/TO/recurrentgemma/flax/2b-it/1`)。
- 模型checkpoint 將位於`/LOCAL/PATH/TO/recurrentgemma/flax/2b-it/1/2b-it`)。

In [ ]:
CKPT_PATH = os.path.join(RECURRENTGEMMA_PATH, RECURRENTGEMMA_VARIANT)
TOKENIZER_PATH = os.path.join(RECURRENTGEMMA_PATH, 'tokenizer.model')
print('CKPT_PATH:', CKPT_PATH)
print('TOKENIZER_PATH:', TOKENIZER_PATH)

CKPT_PATH: /root/.cache/kagglehub/models/google/recurrentgemma/flax/2b-it/1/2b-it
TOKENIZER_PATH: /root/.cache/kagglehub/models/google/recurrentgemma/flax/2b-it/1/tokenizer.model


## 載入並準備 MTNT dataset 和 Gemma tokenizer

您將使用 [MTNT（噪音文字機器翻譯）](https://arxiv.org/abs/1809.00388) dataset，可從 [TensorFlow dataset](https://www.tensorflow.org/datasets/catalog/mtnt) 取得。
下載 MTNT dataset 的英文到法文 dataset 部分，然後取樣兩個範例。 dataset中的每個樣本包含兩個條目：`src`：原始英語句子；和 `dst`：相應的法語翻譯。

In [ ]:
ds = tfds.load("mtnt/en-fr", split="train")

ds = ds.take(2)
ds = ds.as_numpy_iterator()

for idx, example in enumerate(ds):
  print(f'Example {idx}:')
  for key, val in example.items():
    print(f'{key}: {val}')
  print()

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/35692 [00:00<?, ? examples/s]

...

Generating test examples...:   0%|          | 0/1020 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/mtnt/en-fr/1.0.0.incompleteJLH33K/mtnt-test.tfrecord*...:   0%|          |…

Generating valid examples...:   0%|          | 0/811 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/mtnt/en-fr/1.0.0.incompleteJLH33K/mtnt-valid.tfrecord*...:   0%|          …

Dataset mtnt downloaded and prepared to /root/tensorflow_datasets/mtnt/en-fr/1.0.0. Subsequent calls will reuse this data.
Example 0:
dst: b'Le groupe de " toutes les \xc3\xa9toiles potentielles de la conf\xc3\xa9rence de l\'Est mais qui ne s\'en sortent pas dans le groupe de l\'Ouest ".'
src: b'The group of \xe2\x80\x9ceastern conference potential all stars but not making it in the West\xe2\x80\x9d group.'

Example 1:
dst: b"Kameron est-elle un peu aigrie de son manque de temps \xc3\xa0 l'\xc3\xa9cran ?"
src: b'Is Kameron a Little Salty About Her Lack of Air Time?'



載入 Gemma tokenizer，使用 [`sentencepiece.SentencePieceProcessor`](https://github.com/google/sentencepiece/blob/4d6a1f41069c4636c51a5590f7578a0dbed83450/python/src/sentencepiece/__init__.py#L423) 建構：

In [ ]:
vocab = spm.SentencePieceProcessor()
vocab.Load(TOKENIZER_PATH)

True

為英語到法文翻譯任務自訂[`SentencePieceProcessor`](https://github.com/google/sentencepiece/blob/4d6a1f41069c4636c51a5590f7578a0dbed83450/python/src/sentencepiece/__init__.py#L423)。由於您將fine-tuning RecurrentGemma (Griffin) 模型的英語部分，因此您需要進行一些調整，例如：
- *輸入前綴*：為每個輸入新增公共前綴表示翻譯任務。例如，您可以使用帶有`Translate this into French: [INPUT_SENTENCE]` 等前綴的prompt。

- *翻譯開始後綴*：在每個 prompt 末尾添加後綴可以準確指示 Gemma 模型何時開始翻譯過程。一條新線應該可以完成這項工作。

- *語言模型tokens*：循環Gemma (Griffin) 模型期望每個序列的開頭有一個「序列開頭」token。同樣，您需要在每個訓練範例的末尾添加“序列結束”token。

圍繞 `SentencePieceProcessor` 建立自訂包裝器，如下所示：

In [ ]:
class GriffinTokenizer:
  """A custom wrapper around a SentencePieceProcessor."""

  def __init__(self, spm_processor: spm.SentencePieceProcessor):
    self._spm_processor = spm_processor

  @property
  def pad_id(self) -> int:
    """Fast access to the pad ID."""
    return self._spm_processor.pad_id()

  def tokenize(
      self,
      example: str | bytes,
      prefix: str = '',
      suffix: str = '',
      add_eos: bool = True,
  ) -> jax.Array:
    """
    A tokenization function.

    Args:
      example: Input string to tokenize.
      prefix:  Prefix to add to the input string.
      suffix:  Suffix to add to the input string.
      add_eos: If True, add an end of sentence token at the end of the output
               sequence.
    Returns:
      Tokens corresponding to the input string.
    """
    int_list = [self._spm_processor.bos_id()]
    int_list.extend(self._spm_processor.EncodeAsIds(prefix + example + suffix))
    if add_eos:
      int_list.append(self._spm_processor.eos_id())

    return jnp.array(int_list, dtype=jnp.int32)

  def tokenize_tf_op(
      self,
      str_tensor: tf.Tensor,
      prefix: str = '',
      suffix: str = '',
      add_eos: bool = True,
  ) -> tf.Tensor:
    """A TensforFlow operator for the `tokenize` function."""
    encoded = tf.numpy_function(
        self.tokenize,
        [str_tensor, prefix, suffix, add_eos],
        tf.int32)
    encoded.set_shape([None])
    return encoded

  def to_string(self, tokens: jax.Array) -> str:
    """Convert an array of tokens to a string."""
    return self._spm_processor.EncodeIds(tokens.tolist())

透過實例化新的自訂 `GriffinTokenizer` 來嘗試一下，然後將其應用於 MTNT dataset 的小樣本：

In [ ]:
def tokenize_source(tokenizer, example: tf.Tensor):
  return tokenizer.tokenize_tf_op(
      example,
      prefix='Translate this into French:\n',
      suffix='\n',
      add_eos=False
  )
def tokenize_destination(tokenizer, example: tf.Tensor):
  return tokenizer.tokenize_tf_op(example, add_eos=True)

tokenizer = GriffinTokenizer(vocab)

ds = tfds.load("mtnt/en-fr",split="train")
ds = ds.take(2)
ds = ds.map(lambda x: {
    'src': tokenize_source(tokenizer, x['src']),
    'dst': tokenize_destination(tokenizer, x['dst'])
  })
ds = ds.as_numpy_iterator()

for idx, example in enumerate(ds):
  print(f'Example {idx}:')
  for key, val in example.items():
    print(f'{key}: {val}')
  print()

Example 0:
src: [     2  49688    736   1280   6987 235292    108    651   2778    576
   1080 104745  11982   5736    832   8995    901    780   3547    665
    575    573   4589 235369   2778 235265    108]
dst: [     2   2025  29653    581    664  16298   1437  55563  41435   7840
    581    683 111452    581    533 235303   9776   4108   2459    679
    485 235303    479   6728    579   1806   2499    709  29653    581
    533 235303 101323  16054      1]

Example 1:
src: [     2  49688    736   1280   6987 235292    108   2437  87150    477
    476  11709 230461   8045   3636  40268    576   4252   4897 235336
    108]
dst: [     2 213606    477   1455 235290   3510    748   8268 191017   2809
    581   2032  69972    581  11495   1305    533 235303  65978   1654
      1]



為整個 MTNT dataset 建立資料載入器：

In [ ]:
@chex.dataclass(frozen=True)
class TrainingInput:
  # Input tokens provided to the model.
  input_tokens: jax.Array

  # A mask that determines which tokens contribute to the target loss
  # calculation.
  target_mask: jax.Array

class DatasetSplit(enum.Enum):
  TRAIN = 'train'
  VALIDATION = 'valid'


class MTNTDatasetBuilder:
  """A data loader for the MTNT dataset."""

  N_ITEMS = {DatasetSplit.TRAIN: 35_692, DatasetSplit.VALIDATION: 811}

  BUFFER_SIZE_SHUFFLE = 10_000
  TRANSLATION_PREFIX = 'Translate this into French:\n'
  TRANSLATION_SUFFIX = '\n'

  def __init__(self,
               tokenizer : GriffinTokenizer,
               max_seq_len: int):
    """A constructor.

    Args:
      tokenizer: The tokenizer to use.
      max_seq_len: The size of each sequence in a given batch.
    """
    self._tokenizer = tokenizer
    self._base_data = {
        DatasetSplit.TRAIN: tfds.load("mtnt/en-fr",split="train"),
        DatasetSplit.VALIDATION: tfds.load("mtnt/en-fr",split="valid"),
    }
    self._max_seq_len = max_seq_len

  def _tokenize_source(self, example: tf.Tensor):
    """A tokenization function for the source."""
    return self._tokenizer.tokenize_tf_op(
        example, prefix=self.TRANSLATION_PREFIX, suffix=self.TRANSLATION_SUFFIX,
        add_eos=False
    )

  def _tokenize_destination(self, example: tf.Tensor):
    """A tokenization function for the French translation."""
    return self._tokenizer.tokenize_tf_op(example, add_eos=True)

  def _pad_up_to_max_len(self,
                         input_tensor: tf.Tensor,
                         pad_value: int | bool,
                         ) -> tf.Tensor:
    """Pad the given tensor up to sequence length of a batch."""
    seq_len = tf.shape(input_tensor)[0]
    to_pad = tf.maximum(self._max_seq_len - seq_len, 0)
    return tf.pad(
        input_tensor, [[0, to_pad]], mode='CONSTANT', constant_values=pad_value,
    )

  def _to_training_input(
      self,
      src_tokens: jax.Array,
      dst_tokens: jax.Array,
  ) -> TrainingInput:
    """Build a training input from a tuple of source and destination tokens."""

    # The input sequence fed to the model is simply the concatenation of the
    # source and the destination.
    tokens = tf.concat([src_tokens, dst_tokens], axis=0)

    # You want to prevent the model from updating based on the source (input)
    # tokens. To achieve this, add a target mask to each input.
    q_mask = tf.zeros_like(src_tokens, dtype=tf.bool)
    a_mask = tf.ones_like(dst_tokens, dtype=tf.bool)
    mask = tf.concat([q_mask, a_mask], axis=0)

    # If the output tokens sequence is smaller than the target sequence size,
    # then pad it with pad tokens.
    tokens = self._pad_up_to_max_len(tokens, self._tokenizer.pad_id)

    # You don't want to perform the backward on the pad tokens.
    mask = self._pad_up_to_max_len(mask, False)

    return TrainingInput(input_tokens=tokens, target_mask=mask)


  def get_train_dataset(self, batch_size: int, num_epochs: int):
    """Build the training dataset."""

    # Tokenize each sample.
    ds = self._base_data[DatasetSplit.TRAIN].map(
        lambda x : (self._tokenize_source(x['src']),
                    self._tokenize_destination(x['dst']))
    )

    # Convert them to training inputs.
    ds = ds.map(lambda x, y: self._to_training_input(x, y))

    # Remove the samples which are too long.
    ds = ds.filter(lambda x: tf.shape(x.input_tokens)[0] <= self._max_seq_len)

    # Shuffle the dataset.
    ds = ds.shuffle(buffer_size=self.BUFFER_SIZE_SHUFFLE)

    # Repeat if necessary.
    ds = ds.repeat(num_epochs)

    # Build batches.
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

  def get_validation_dataset(self, batch_size: int):
    """Build the validation dataset."""

    # Same as the training dataset, but no shuffling and no repetition
    ds = self._base_data[DatasetSplit.VALIDATION].map(
        lambda x : (self._tokenize_source(x['src']),
                    self._tokenize_destination(x['dst']))
    )
    ds = ds.map(lambda x, y: self._to_training_input(x, y))
    ds = ds.filter(lambda x: tf.shape(x.input_tokens)[0] <= self._max_seq_len)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

透過再次實例化自訂 `GriffinTokenizer` 來嘗試 `MTNTDatasetBuilder`，然後將其應用到 MTNT dataset 上，並取樣兩個範例：

In [ ]:
dataset_builder = MTNTDatasetBuilder(tokenizer, max_seq_len=20)
ds = dataset_builder.get_train_dataset(3, 1)
ds = ds.take(2)
ds = ds.as_numpy_iterator()

for idx, example in enumerate(ds):
  print(f'Example {idx}:')
  for key, val in example.items():
    print(f'{key}: {val}')
  print()

Example 0:
input_tokens: [[     2  49688    736   1280   6987 235292    108  12583    665 235265
     108      2   6151  94975   1320   6238 235265      1      0      0]
 [     2  49688    736   1280   6987 235292    108   4899  29960  11270
  108282 235265    108      2   4899  79025  11270 108282      1      0]
 [     2  49688    736   1280   6987 235292    108  26620 235265    108
       2  26620 235265      1      0      0      0      0      0      0]]
target_mask: [[False False False False False False False False False False False  True
   True  True  True  True  True  True False False]
 [False False False False False False False False False False False False
  False  True  True  True  True  True  True False]
 [False False False False False False False False False False  True  True
   True  True False False False False False False]]

Example 1:
input_tokens: [[     2  49688    736   1280   6987 235292    108    527   5174   1683
  235336    108      2 206790    581  20726    482  

## 設定模型

在開始 fine-tuning Gemma 模型之前，您需要對其進行設定。
使用 [`recurrentgemma.jax.utils.load_parameters`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/utils.py#L31) 方法載入 RecurrentGemma (Griffin) 模型checkpoint：

In [ ]:
params =  recurrentgemma.load_parameters(CKPT_PATH, "single_device")

若要從 RecurrentGemma 型號 checkpoint 自動載入正確的設定，請使用 [`recurrentgemma.GriffinConfig.from_flax_params_or_variables`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/common.py#L128)：

In [ ]:
config = recurrentgemma.GriffinConfig.from_flax_params_or_variables(params)

使用 [`recurrentgemma.jax.Griffin`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/griffin.py#L29) 實例化 [Griffin](https://arxiv.org/abs/2402.19427) 模型：

In [ ]:
model = recurrentgemma.Griffin(config)

在 RecurrentGemma 模型 checkpoint/weights 和 tokenizer 之上創建一個帶有 [`recurrentgemma.jax.Sampler`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/sampler.py#L74) 的 `sampler` 以檢查您的模型是否可以執行翻譯：

In [ ]:
sampler = recurrentgemma.Sampler(model=model, vocab=vocab, params=params)

## 微調模型

在本節中，您將：
- 使用`gemma.transformer.Transformer`類別建立前向傳遞和損失函數。
- 為 tokens 建構位置和注意掩模向量
- 使用 Flax 建立訓練步驟函數。
- 建置驗證步驟，無需向後傳遞。
- 創建訓練循環。
- 微調 Gemma 模型。

Define the forward pass and the loss function using the [`recurrentgemma.jax.griffin.Griffin`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/griffin.py#L29) 類。 The RecurrentGemma `Griffin` inherits from [`flax.linen.Module`](https://flax.readthedocs.io/en/latest/api_reference/flax.linen/module.html), and offers two essential methods:
- `init`：初始化模型的參數。
- `apply`：使用給定的參數集執行模型的`__call__` 函數。

由於您使用的是預先訓練的 Gemma 權重，因此不需要使用 `init` 函數。

In [ ]:
def forward_and_loss_fn(
    params,
    *,
    model: recurrentgemma.Griffin,
    input_tokens: jax.Array,            # Shape [B, L]
    input_mask: jax.Array,              # Shape [B, L]
    positions: jax.Array,               # Shape [B, L]
) -> jax.Array:
  """Forward pass and loss function.

  Args:
    params: model's input parameters.
    model: Griffin model to call.
    input_tokens: input tokens sequence, shape [B, L].
    input_mask: tokens to ignore when computing the loss, shape [B, L].
    positions: relative position of each token, shape [B, L].

  Returns:
    Softmax cross-entropy loss for the next-token prediction task.
  """
  batch_size = input_tokens.shape[0]
  # Forward pass on the input data.
  # No attention cache is needed here.
  # Exclude the last step as it does not appear in the targets.
  logits, _ = model.apply(
        {"params": params},
        tokens=input_tokens[:, :-1],
        segment_pos=positions[:, :-1],
        cache=None,
    )

  # Similarly, the first token cannot be predicteds.
  target_tokens = input_tokens[:, 1:]
  target_mask = input_mask[:, 1:]

  # Convert the target labels into one-hot encoded vectors.
  one_hot = jax.nn.one_hot(target_tokens, logits.shape[-1])

  # Don't update on unwanted tokens.
  one_hot = one_hot * target_mask.astype(one_hot.dtype)[...,None]

  # Normalization factor.
  norm_factor = batch_size * (jnp.sum(target_mask) + 1e-8)

  # Return the negative log-likelihood loss (NLL) function.
  return -jnp.sum(jax.nn.log_softmax(logits) * one_hot) / norm_factor

建立執行向後傳遞並相應更新模型參數的 `train_step` 函數，其中：
- [`jax.value_and_grad`](https://jax.readthedocs.io/en/latest/_autosummary/jax.value_and_grad.html) 用於評估前向和後向傳遞過程中的損失函數和梯度。
- [`optax.apply_updates`](https://optax.readthedocs.io/en/latest/api/apply_updates.html#optax.apply_updates) 用於更新參數。

In [ ]:
Params = Mapping[str, Any]

def get_positions(example: jax.Array, pad_id : int) -> jax.Array:
  """Builds the position vector from the given tokens."""
  pad_mask = example != pad_id
  positions = jnp.cumsum(pad_mask, axis=-1)
  # Subtract one for all positions from the first valid one as they are
  # 0-indexed
  positions = positions - (positions >= 1)
  return positions

@functools.partial(
    jax.jit,
    static_argnames=['model', 'optimizer'],
    donate_argnames=['params', 'opt_state'],
)
def train_step(
    model: recurrentgemma.Griffin,
    params: Params,
    optimizer: optax.GradientTransformation,
    opt_state: optax.OptState,
    pad_id: int,
    example: TrainingInput,
) -> tuple[jax.Array, Params, optax.OptState]:
  """The train step.

  Args:
    model: The RecurrentGemma (Griffin) model.
    params: The model's input parameters.
    optimizer: The Optax optimizer to use.
    opt_state: The input optimizer's state.
    pad_id: The ID of the pad token.
    example: The input batch.

  Returns:
    Training loss, updated parameters, updated optimizer state.
  """

  positions = get_positions(example.input_tokens, pad_id)

  # Forward and backward passes.
  train_loss, grads = jax.value_and_grad(forward_and_loss_fn)(
      params,
      model=model,
      input_tokens=example.input_tokens,
      input_mask=example.target_mask,
      positions=positions,
  )
  # Update the parameters.
  updates, opt_state = optimizer.update(grads, opt_state, params)
  params = optax.apply_updates(params, updates)

  return train_loss, params, opt_state

建立 `validation_step` 函數，無需向後傳遞：

In [ ]:
@functools.partial(jax.jit, static_argnames=['model'])
def validation_step(
    model: recurrentgemma.Griffin,
    params: Params,
    pad_id: int,
    example: TrainingInput,
) -> jax.Array:
  return forward_and_loss_fn(
      params,
      model=model,
      input_tokens=example.input_tokens,
      input_mask=example.target_mask,
      positions=get_positions(example.input_tokens, pad_id),
  )

定義訓練循環：

In [ ]:
def train_loop(
    model: recurrentgemma.Griffin,
    params: Params,
    optimizer: optax.GradientTransformation,
    train_ds: Iterator[TrainingInput],
    validation_ds: Iterator[TrainingInput],
    num_steps: int | None = None,
    eval_every_n: int = 20,
):
  opt_state = jax.jit(optimizer.init)(params)

  step_counter = 0
  avg_loss=0

  # The first round of the validation loss.
  n_steps_eval = 0
  eval_loss = 0
  for val_example in validation_ds.as_numpy_iterator():
    eval_loss += validation_step(
        model, params, dataset_builder._tokenizer.pad_id, val_example
    )
    n_steps_eval += 1
  print(f"Start, validation loss: {eval_loss/n_steps_eval}")

  for train_example in train_ds:
    train_loss, params, opt_state = train_step(
        model=model,
        params=params,
        optimizer=optimizer,
        opt_state=opt_state,
        pad_id=dataset_builder._tokenizer.pad_id,
        example=train_example,
    )

    step_counter += 1
    avg_loss += train_loss
    if step_counter % eval_every_n == 0:
      eval_loss = 0

      n_steps_eval = 0
      val_iterator = validation_ds.as_numpy_iterator()
      for val_example in val_iterator:
        eval_loss += validation_step(
            model,
            params,
            dataset_builder._tokenizer.pad_id,
            val_example,
        )
        n_steps_eval +=1
      avg_loss /= eval_every_n
      eval_loss /= n_steps_eval
      print(f"STEP {step_counter} training loss: {avg_loss} - eval loss: {eval_loss}")
      avg_loss=0
    if num_steps is not None and step_counter > num_steps:
      break
  return params

在這裡你必須選擇一個（Optax）優化器。對於記憶體較小的設備，您應該使用 SGD，因為它的記憶體佔用量要低得多。要獲得最佳 fine-tuning 性能，請嘗試 Adam-W。此範例中為 `2b-it` checkpoint 提供了針對此 notebook 中的特定任務的每個最佳化器的最佳超參數。

In [ ]:
def griffin_weight_decay_mask(params_like: optax.Params) -> Any:
  # Don't put weight decay on the RGLRU, the embeddings and any biases
  def enable_weight_decay(path: list[Any], _: Any) -> bool:
    # Parameters in the LRU and embedder
    path = [dict_key.key for dict_key in path]
    if 'rg_lru' in path or 'embedder' in path:
      return False
    # All biases and scales
    if path[-1] in ('b', 'scale'):
      return False
    return True

  return jax.tree_util.tree_map_with_path(enable_weight_decay, params_like)

optimizer_choice = "sgd" #@param ["sgd", "adamw"]

if optimizer_choice == "sgd":
  optimizer = optax.sgd(learning_rate=1e-3)
  num_steps = 300
elif optimizer_choice == "adamw":
  optimizer = optax.adamw(
        learning_rate=1e-4,
        b2=0.96,
        eps=1e-8,
        weight_decay=0.1,
        mask=griffin_weight_decay_mask,
    )
  num_steps = 100
else:
  raise ValueError(f"Unknown optimizer: {optimizer_choice}")

準備訓練和驗證datasets：

In [ ]:
# Choose a small sequence length size, so that everything fits in memory.
num_epochs = 1 #@param {type: "integer"}
batch_size = 1 #@param {type: "integer"}
sequence_length = 32 #@param {type: "integer"}

# Make the dataset builder.
tokenizer = GriffinTokenizer(vocab)
dataset_builder= MTNTDatasetBuilder(tokenizer, sequence_length + 1)

# Build the training dataset.
train_ds = dataset_builder.get_train_dataset(
    batch_size=batch_size,
    num_epochs=num_epochs,
).as_numpy_iterator()

# Build the validation dataset, with a limited number of samples for this demo.
validation_ds = dataset_builder.get_validation_dataset(
    batch_size=batch_size,
).take(50)

在有限數量的步驟上開始 fine-tuning 循環Gemma (Griffin) 模型 (`num_steps`)：

In [ ]:
trained_params = train_loop(
    model=model,
    params=params,
    optimizer=optimizer,
    train_ds=train_ds,
    validation_ds=validation_ds,
    num_steps=num_steps,
)

Start, validation loss: 7.894117832183838


/usr/local/lib/python3.10/dist-packages/jax/_src/interpreters/mlir.py:920: UserWarning: Some donated buffers were not usable: ShapedArray(int32[1,33]), ShapedArray(bool[1,33]), ShapedArray(int32[], weak_type=True).
See an explanation at https://jax.readthedocs.io/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"


STEP 20 training loss: 4.592616081237793 - eval loss: 2.847407102584839
STEP 40 training loss: 2.7537424564361572 - eval loss: 2.9258534908294678
STEP 60 training loss: 2.835618257522583 - eval loss: 2.4382340908050537
STEP 80 training loss: 2.6322107315063477 - eval loss: 2.3696839809417725
STEP 100 training loss: 1.8703256845474243 - eval loss: 2.355681896209717
STEP 120 training loss: 2.7280433177948 - eval loss: 2.4059958457946777
STEP 140 training loss: 2.3047447204589844 - eval loss: 2.083082914352417
STEP 160 training loss: 2.3432137966156006 - eval loss: 2.095074415206909
STEP 180 training loss: 2.1081202030181885 - eval loss: 2.006460189819336
STEP 200 training loss: 2.5359647274017334 - eval loss: 1.9667452573776245
STEP 220 training loss: 2.202195644378662 - eval loss: 1.9440618753433228
STEP 240 training loss: 2.756615400314331 - eval loss: 2.1073737144470215
STEP 260 training loss: 2.5128934383392334 - eval loss: 2.117241859436035
STEP 280 training loss: 2.73045015335083 -

訓練損失和驗證損失都應該隨著步數的增加而下降。
為了確保您的輸入與訓練格式匹配，請記住使用前綴 `Translate this into French:\n` 並在末尾使用換行符。这标志着模型开始翻译。

In [ ]:
sampler.params = trained_params
output = sampler(
    ["Translate this into French:\nHello, my name is Morgane.\n"],
    total_generation_steps=100,
)
print(output.text[0])

/usr/local/lib/python3.10/dist-packages/jax/_src/interpreters/mlir.py:920: UserWarning: Some donated buffers were not usable: ShapedArray(int32[1,16]).
See an explanation at https://jax.readthedocs.io/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"


Mais je m'appelle Morgane.


## 了解更多

- 您可以了解有關 Google DeepMind [`recurrentgemma` library on GitHub](https://github.com/google-deepmind/recurrentgemma) 的更多信息，其中包含您在本教學中使用的方法和模組的文檔字符串，例如 [`recurrentgemma.jax.load_parameters`](@@P0050@P0050和[`recurrentgemma.jax.Sampler`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/sampler.py#L74)。
- 以下函式庫有自己的文件網站：[core JAX](https://jax.readthedocs.io)、[Flax](https://flax.readthedocs.io)、[Chex](https://chex.readthedocs.io/en/latest/)、[Optax](https://optax.readthedocs.io/en/latest/P0003@Pba00(4)。
- 有關 `sentencepiece` tokenizer/detokenizer 文檔，請查看 [Google 的 `sentencepiece` GitHub 儲存庫](https://github.com/google/sentencepiece)。
- 對於 `kagglehub` 文檔，請查看 [Kaggle 的 `kagglehub` GitHub 存儲庫](https://github.com/Kaggle/kagglehub) 上的 `README.md`。
- 了解如何[將 Gemma 模型與 Google Cloud Vertex AI 一起使用](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma)。
- 如果您使用 Google Cloud TPU（v3-8 及更高版本），請確保同時更新至最新的 `jax[tpu]` 軟體包 (`!pip install -U jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html`)，重新啟動 runtime，並檢查 `jax` 和 @@P0003@P 版本是否符合 (4@P.004@P0003@P 版本。這可以防止由於`jaxlib` 和`jax` 版本不匹配而出現`RuntimeError`。有關JAX的更多安裝說明，請參閱[JAX文件](https://jax.readthedocs.io/en/latest/tutorials/installation.html#install-google-tpu)。
- 查看[循環Gemma：過去Transformers
高效開放語言模型](https://arxiv.org/pdf/2404.07839) Google DeepMind 的論文。- 閱讀 [Griffin：將門控線性遞歸與
Local Attention for Efficient Language Models](https://arxiv.org/pdf/2402.19427) Google DeepMind 的論文，以了解有關 RecurrentGemma 使用的模型架構的更多資訊。